# 🧬 Invivo Partners — Biotech Research Agent
### Investment-grade research reports for any disease or treatment

**How to use:**
1. Run **Cell 1** once (installs everything — takes ~30 seconds the first time)
2. Run **Cell 2** to enter your disease and generate the report
3. The report saves as an HTML file you can open in any browser

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Setup (run once per session, takes ~30 seconds)
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os

print('⏳ Installing dependencies...')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'langgraph', 'langchain-anthropic', 'langchain-core', 'anthropic',
     'aiohttp', 'fastapi', 'uvicorn', 'python-multipart', 'jinja2',
     'matplotlib', 'seaborn', 'faiss-cpu', 'ipywidgets'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Make sure the agent package is importable
sys.path.insert(0, '.')
os.environ.setdefault('MPLBACKEND', 'Agg')
os.makedirs('outputs', exist_ok=True)

print('✅ Ready! Run Cell 2 to generate a report.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Generate a report (run this cell each time)
# ═══════════════════════════════════════════════════════════════

import sys, os, time
sys.path.insert(0, '.')
os.environ.setdefault('MPLBACKEND', 'Agg')

import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output
from biotech_agent.pipeline import run_sync

# ── Input box ────────────────────────────────────────────────
disease_input = widgets.Text(
    placeholder='e.g. ALS, pancreatic cancer, atopic dermatitis...',
    description='Disease:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '80px'}
)

run_button = widgets.Button(
    description='Generate Report',
    button_style='success',
    icon='flask',
    layout=widgets.Layout(width='180px', height='36px')
)

output_area = widgets.Output()

def on_run_clicked(b):
    disease = disease_input.value.strip()
    if not disease:
        with output_area:
            clear_output()
            print('⚠️  Please enter a disease name first.')
        return

    with output_area:
        clear_output(wait=True)
        print(f'🔬 Generating report for: {disease}')
        print('─' * 50)

        start = time.time()
        steps = []

        def progress(msg):
            steps.append(msg)
            print(f'  {msg}')

        try:
            result = run_sync(disease, progress_callback=progress)
            elapsed = time.time() - start
            html_content = result['html']
            matched = result['retrieval_stats'].get('matched_disease', 'live search')
            size_kb = len(html_content) // 1024

            # Save report
            safe_name = disease.lower().replace(' ', '_').replace('/', '_')[:40]
            output_path = f'outputs/{safe_name}_report.html'
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(html_content)

            print('─' * 50)
            print(f'✅ Done in {elapsed:.1f}s  |  {size_kb}KB  |  database: {matched}')
            print()
            display(HTML(f'''
                <div style="background:#f0fff8;border:1px solid #1D9E75;border-radius:8px;padding:16px;margin-top:8px">
                    <b>📄 Report ready:</b> {output_path}<br><br>
                    <b>To open it:</b><br>
                    &nbsp;&nbsp;• <b>Codespaces:</b> right-click the file in the left panel → Open with Live Server, or double-click to preview<br>
                    &nbsp;&nbsp;• <b>Local Jupyter:</b> the file is in your <code>outputs/</code> folder — open it in your browser
                </div>
            '''))
            display(FileLink(output_path, result_html_prefix='⬇️  Download: '))

        except Exception as e:
            print(f'❌ Error: {e}')
            import traceback; traceback.print_exc()

run_button.on_click(on_run_clicked)

# ── Display the UI ────────────────────────────────────────────
display(HTML('<h3 style="color:#1D9E75;margin-bottom:4px">🧬 Biotech Research Agent</h3>'))
display(HTML('<p style="color:#666;margin-bottom:12px">Enter a disease, drug name, or treatment to generate an investor report.</p>'))
display(widgets.HBox([disease_input, run_button]))
display(HTML('<p style="color:#999;font-size:12px;margin-top:6px">'
    'Works without API key: Alzheimer · ALS · KRAS · GLP-1/obesity · Atopic dermatitis · '
    'Pancreatic cancer · NASH · Sickle cell · Multiple sclerosis · CAR-T · Glioblastoma · SMA · RA<br>'
    'Add ANTHROPIC_API_KEY for any other disease.</p>'))
display(output_area)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Preview the last report inside the notebook
# ═══════════════════════════════════════════════════════════════
# Run this after Cell 2 to see the report inline.
# Tip: right-click the output → Open in New Tab for full screen.

import glob, os
from IPython.display import IFrame

reports = sorted(glob.glob('outputs/*_report.html'), key=os.path.getmtime, reverse=True)
if reports:
    print(f'Showing: {reports[0]}')
    display(IFrame(src=reports[0], width='100%', height='950px'))
else:
    print('No reports yet — run Cell 2 first.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Add your API key (unlocks ALL diseases)
# ═══════════════════════════════════════════════════════════════
# Get your key at console.anthropic.com → API Keys
# Paste it below and run this cell — then re-run Cell 2.

import os

# Paste your key here between the quotes:
os.environ['ANTHROPIC_API_KEY'] = ''

if os.environ.get('ANTHROPIC_API_KEY'):
    print('✅ API key set — any disease now works with live PubMed + ClinicalTrials.gov data')
else:
    print('ℹ️  No key entered — 13 curated diseases still work perfectly')